# Qualité et résultats — momentum de clôture

Notebook de synthèse exécuté le 20 octobre 2026. Les scripts indépendants réalisent les contrôles détaillés ; ce notebook expose les preuves et les résultats publiés sans lire de clé ou contacter un compte. B est un historique exploratoire réutilisé.

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
out = Path("../results/2026-10-20-v2")
profile = json.loads((out / "DATA_SOURCE_PROFILE.json").read_text())
print(pd.DataFrame(profile["profiles"])[["symbol", "first", "last", "calendar_sessions", "complete_sessions", "calendar_continuous"]].to_string(index=False))

symbol      first       last  calendar_sessions  complete_sessions  calendar_continuous
   SPY 2015-12-31 2026-10-16               2714               2705                 True
   QQQ 2015-12-31 2026-10-16               2714               2707                 True


In [2]:
for symbol in ["SPY", "QQQ"]:
    quality = pd.read_csv(out / f"{symbol}_quality.csv")
    quality = quality[quality.date >= "2016-01-04"]
    print(symbol, quality.eligibility.value_counts().to_dict())

SPY {'eligible': 2706, 'invalid_morning_window': 4, 'invalid_previous_close': 2, 'invalid_sizing_minute': 1}
QQQ {'eligible': 2709, 'invalid_morning_window': 3, 'invalid_previous_close': 1}


In [3]:
comparison = pd.read_csv(out / "comparison_all.csv", float_precision="round_trip")
main = comparison[(comparison["mode"] == "momentum") & (comparison.slippage == .01)]
print(main[["symbol", "period", "trades", "total_return_pct", "observed_close_drawdown_pct"]].to_string(index=False))

symbol period  trades  total_return_pct  observed_close_drawdown_pct
   SPY    ALL    2701        -34.722159                    36.714818
   SPY      A    2085        -32.422388                    33.680875
   SPY      B     616         -3.391280                     8.225875
   QQQ    ALL    2705        -35.194933                    38.697974
   QQQ      A    2087        -32.732130                    33.854924
   QQQ      B     618         -3.702651                    10.558339


In [4]:
for filename in ["AUDIT_INDEPENDANT.json", "AGGREGATE_VALIDATION.json", "RAW_EXCERPT_AUDIT.json", "VERSION_COMPARISON.json"]:
    proof = json.loads((out / filename).read_text())
    assert proof["status"] == "passed"
    print(filename, proof["status"])
manifest = json.loads((out / "DELIVERY_MANIFEST.json").read_text())
for filename, expected in manifest["files"].items():
    assert hashlib.sha256((out / filename).read_bytes()).hexdigest() == expected, filename
print("Empreintes du manifeste de livraison concordantes :", len(manifest["files"]))

AUDIT_INDEPENDANT.json passed
AGGREGATE_VALIDATION.json passed
RAW_EXCERPT_AUDIT.json passed
VERSION_COMPARISON.json passed
Empreintes du manifeste de livraison concordantes : 284
